In [ ]:
import tkinter as tk
from tkinter import ttk
import heapq
import random
import math
import time

# ======================
# SETTINGS
# ======================
CELL = 60
DELAY = 120
DIRS = [(-1, 0), (0, 1), (1, 0), (1, 1), (0, -1), (-1, -1)]
ROWS, COLS = 10, 10
grid = [[0]*COLS for _ in range(ROWS)]
start = (0,0)
target = (ROWS-1,COLS-1)
mode = None
running = False
dynamic_mode = False
algo_running = False
MAX_CANVAS_SIZE = 600

# ======================
# COLORS
# ======================
C = {
    "free":"#eeeeee",
    "wall":"#111111",
    "start":"#0b6623",       
    "target":"#8b0000",   
    "explored":"#3366cc",   
    "visited":"#ffcc33",     
    "path":"#00cc00"        
}

# ======================
# HEURISTICS
# ======================
def manhattan(a,b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def euclidean(a,b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def get_heuristic():
    return manhattan if heuristic_box.get()=="Manhattan" else euclidean

# ======================
# NEIGHBORS
# ======================
def neighbors(r,c):
    for dr,dc in DIRS:
        nr,nc = r+dr, c+dc
        if 0<=nr<ROWS and 0<=nc<COLS and grid[nr][nc]!=-1:
            yield nr,nc

# ======================
# DRAW GRID
# ======================
def draw(path=None, explored=None, visited=None, expanded_order=None):
    canvas.delete("all")
    path = path or []
    explored = explored or set()
    visited = visited or set()
    expanded_order = expanded_order or {}

    for r in range(ROWS):
        for c in range(COLS):
            color = C["free"]
            if grid[r][c] == -1:
                color = C["wall"]
            elif (r,c) in explored:
                color = C["explored"]
            elif (r,c) in visited:
                color = C["visited"]
            if (r,c) in path:
                color = C["path"]
            if (r,c)==start:
                color = C["start"]
            if (r,c)==target:
                color = C["target"]

            x1,y1 = c*CELL, r*CELL
            x2,y2 = x1+CELL, y1+CELL
            canvas.create_rectangle(x1,y1,x2,y2, fill=color, outline="gray")

            if (r,c) in expanded_order:
                canvas.create_text(x1+CELL//2, y1+CELL//2,text=str(expanded_order[(r,c)]),fill="white", font=("Arial", 12, "bold"))
    draw_legend()

# ======================
# LEGEND
# ======================
def draw_legend():
    x0, y0 = COLS*CELL + 10, 10
    spacing = 30
    legend_items = [
        ("Free", C["free"]),
        ("Wall", C["wall"]),
        ("Start", C["start"]),
        ("Target", C["target"]),
        ("Explored", C["explored"]),
        ("Visited", C["visited"]),
        ("Path", C["path"])
    ]
    for i,(key,color) in enumerate(legend_items):
        canvas.create_rectangle(x0, y0+i*spacing, x0+20, y0+i*spacing+20, fill=color)
        canvas.create_text(x0+60, y0+i*spacing+10, text=key, anchor="w", font=("Arial",12))

# ======================
# RECONSTRUCT PATH
# ======================
def reconstruct(parent,node):
    path=[]
    while node in parent:
        path.append(node)
        node = parent[node]
    path.append(start)
    return path[::-1]

# ======================
# CLICK HANDLER
# ======================
def click(event):
    global start,target
    if running: return
    r = event.y//CELL
    c = event.x//CELL
    if r>=ROWS or c>=COLS: return
    if mode=="start":
        start=(r,c)
        goal_status_label.config(text="Goal Status: In Process...")
    elif mode=="target":
        target=(r,c)
        goal_status_label.config(text="Goal Status: In Process...")
    elif mode=="wall":
        grid[r][c] = -1 if grid[r][c]==0 else 0
    draw()

def set_mode(m):
    global mode
    mode = m

# ======================
# DYNAMIC MODE TOGGLE
# ======================
def toggle_dynamic_mode():
    global dynamic_mode
    dynamic_mode = not dynamic_mode
    btn_dynamic.config(text=f"Dynamic Mode: {'ON' if dynamic_mode else 'OFF'}")


# start_time = 0
# ======================
# STEPWISE SEARCH
# ======================
def step_search():
    global running, pq, g_cost, parent, explored
    global expanded_order, current_path, algo_running, start_time

    if not algo_running:
        algo = algo_box.get()
        h = get_heuristic()
        if algo=="Greedy":
            pq = [(h(start,target), start)]
            g_cost = {}
        else:
            pq = [(0,start)]
            g_cost = {start:0}
        parent = {}
        explored = set()
        expanded_order = {}
        current_path = []
        algo_running = True
        start_time = time.time()
        resize_btn.config(state="disabled") 
        goal_status_label.config(text="Goal Status: In Process...")  

    if not pq:
        finish_search()
        return

    _, current = heapq.heappop(pq)
    if current in explored:
        root.after(DELAY, step_search)
        return

    explored.add(current)
    expanded_order[current] = len(expanded_order)+1

    # STOP at goal
    if current == target:
        current_path[:] = reconstruct(parent, current)
        finish_search()
        return

    algo = algo_box.get()
    h = get_heuristic()
    for nb in neighbors(*current):
        if nb not in explored:
            if algo=="Greedy":
                heapq.heappush(pq,(h(nb,target), nb))
                parent[nb] = current
            else:
                temp_g = g_cost[current]+1
                if nb not in g_cost or temp_g < g_cost[nb]:
                    g_cost[nb]=temp_g
                    f = temp_g + h(nb,target)
                    heapq.heappush(pq,(f,nb))
                    parent[nb]=current

    # Dynamic obstacles
    if dynamic_mode:
        for r in range(ROWS):
            for c in range(COLS):
                if (r,c)!=start and (r,c)!=target and (r,c) not in current_path:
                    if random.random()<0.03:
                        grid[r][c]=-1

    nodes_visited_label.config(text=f"Nodes Visited: {len(explored)}")
    path_cost_label.config(text=f"Path Cost: {len(current_path)}")
    exec_time_label.config(text=f"Execution Time: {int((time.time()-start_time)*1000)} ms")

    draw(path=current_path, explored=explored, visited={n for _,n in pq}, expanded_order=expanded_order)
    root.after(DELAY, step_search)

# ======================
# Update in finish_search
# ======================
def finish_search():
    global running, algo_running
    running=False
    algo_running=False
    resize_btn.config(state="normal")  # enable resize
    nodes_visited_label.config(text=f"Nodes Visited: {len(explored)}")
    path_cost_label.config(text=f"Path Cost: {len(current_path)}")
    exec_time_label.config(text=f"Execution Time: {int((time.time()-start_time)*1000)} ms")
    
    if current_path and current_path[-1]==target:
        goal_status_label.config(text="Goal Status: Goal Reached!")
    else:
        goal_status_label.config(text="Goal Status: Path Not Found!")
    
    draw(path=current_path, explored=explored, visited=set(), expanded_order=expanded_order)

# ======================
# RUN BUTTON
# ======================
def run_dynamic():
    global running, algo_running
    if running: return
    running=True
    algo_running=False
    step_search()

# ======================
# RESIZE GRID
# ======================


def resize_grid():
    global ROWS, COLS, grid, start, target, CELL, FONT_SIZE
    if running: return
    try:
        ROWS = int(rows_entry.get())
        COLS = int(cols_entry.get())
    except:
        return

    # Dynamically calculate cell size
    CELL = min(MAX_CANVAS_SIZE // ROWS, MAX_CANVAS_SIZE // COLS)

    # Adjust font size dynamically based on CELL
    FONT_SIZE = max(CELL // 5, 5)  # minimum font size 5

    grid = [[0]*COLS for _ in range(ROWS)]
    start = (0,0)
    target = (ROWS-1, COLS-1)

    canvas.config(width=COLS*CELL + 150, height=ROWS*CELL)
    draw()

# ======================
# RANDOM MAP
# ======================
def generate_random_map():
    try:
        density=int(density_entry.get())
        if not 0<=density<=100: return
    except: return
    for r in range(ROWS):
        for c in range(COLS):
            if (r,c)==start or (r,c)==target:
                grid[r][c]=0
            else:
                grid[r][c] = -1 if random.randint(1,100)<=density else 0
    draw()

# ======================
# GUI SETUP
# ======================
root = tk.Tk()
root.title("Dynamic Grid Pathfinder with Metrics")

canvas = tk.Canvas(root, width=COLS*CELL+150, height=ROWS*CELL)
canvas.grid(row=0,column=0,rowspan=20)

panel = tk.Frame(root)
panel.grid(row=0,column=1,padx=10, sticky="n")

# Grid size
tk.Label(panel,text="Rows:").pack(pady=2)
rows_entry = tk.Entry(panel); rows_entry.insert(0,str(ROWS)); rows_entry.pack(pady=2)
tk.Label(panel,text="Cols:").pack(pady=2)
cols_entry = tk.Entry(panel); cols_entry.insert(0,str(COLS)); cols_entry.pack(pady=2)
resize_btn = tk.Button(panel,text="Resize Grid",command=resize_grid); resize_btn.pack(fill="x", pady=2)

# Obstacle
tk.Label(panel,text="Obstacle %:").pack(pady=2)
density_entry = tk.Entry(panel); density_entry.insert(0,"30"); density_entry.pack(pady=2)
tk.Button(panel,text="Random Map",command=generate_random_map).pack(fill="x", pady=2)

# Algorithm & Heuristic
tk.Label(panel,text="Algorithm").pack(pady=2)
algo_box = ttk.Combobox(panel, values=["Greedy","A*"]); algo_box.set("A*"); algo_box.pack(pady=2)
tk.Label(panel,text="Heuristic").pack(pady=2)
heuristic_box = ttk.Combobox(panel, values=["Manhattan","Euclidean"]); heuristic_box.set("Manhattan"); heuristic_box.pack(pady=2)

# Mode buttons
tk.Button(panel,text="Set Start",command=lambda:set_mode("start"), bg="#90ee90").pack(fill="x", pady=3)
tk.Button(panel,text="Set Target",command=lambda:set_mode("target"), bg="#ff7f7f").pack(fill="x", pady=3)
tk.Button(panel,text="Toggle Wall",command=lambda:set_mode("wall")).pack(fill="x", pady=3)

# Dynamic mode
btn_dynamic = tk.Button(panel, text="Dynamic Mode: OFF", command=toggle_dynamic_mode)
btn_dynamic.pack(fill="x", pady=3)

# Run button above metrics
tk.Button(panel,text="Run", command=run_dynamic).pack(fill="x", pady=5)

# Metrics
tk.Label(panel,text="Metrics", font=("Arial",14,"bold")).pack(pady=3)
nodes_visited_label = tk.Label(panel,text="Nodes Visited: 0"); nodes_visited_label.pack(pady=1)
path_cost_label = tk.Label(panel,text="Path Cost: 0"); path_cost_label.pack(pady=1)
exec_time_label = tk.Label(panel,text="Execution Time: 0 ms"); exec_time_label.pack(pady=1)
goal_status_label = tk.Label(panel,text="Goal Status: N/A"); goal_status_label.pack(pady=1)



canvas.bind("<Button-1>", click)
draw()
root.mainloop()